# 自注意力和位置编码

注意力机制中，每个查询都会关注所有的键－值对并生成一个注意力输出。

由于查询、键和值来自同一组输入，因此被称为
自注意力。

使用自注意力进行序列编码，以及如何使用序列的顺序作为补充信息。

In [2]:
import math
import torch
from torch import nn

自注意力

给定一个由词元组成的输入序列$\mathbf{x}_1, \ldots, \mathbf{x}_n$，
其中任意$\mathbf{x}_i \in \mathbb{R}^d$（$1 \leq i \leq n$）。
该序列的自注意力输出为一个长度相同的序列
$\mathbf{y}_1, \ldots, \mathbf{y}_n$，其中：

$$\mathbf{y}_i = f(\mathbf{x}_i, (\mathbf{x}_1, \mathbf{x}_1), \ldots, (\mathbf{x}_n, \mathbf{x}_n)) \in \mathbb{R}^d$$

下面的代码片段是基于多头注意力对一个张量完成自注意力的计算

In [3]:
import math
import torch
from torch import nn

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == self.embed_dim, "embed_dim must be divisible by num_heads"

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]

        # Project query, key, value
        q = self.q_proj(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        output = torch.matmul(attention_weights, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.embed_dim)

        return self.out_proj(output)




MultiHeadAttention(
  (q_proj): Linear(in_features=100, out_features=100, bias=True)
  (k_proj): Linear(in_features=100, out_features=100, bias=True)
  (v_proj): Linear(in_features=100, out_features=100, bias=True)
  (out_proj): Linear(in_features=100, out_features=100, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [4]:
num_hiddens, num_heads = 100, 5
# Assuming query_size, key_size, value_size are all num_hiddens for self-attention
attention = MultiHeadAttention(num_hiddens, num_heads, dropout=0.5)
attention.eval()

MultiHeadAttention(
  (q_proj): Linear(in_features=100, out_features=100, bias=True)
  (k_proj): Linear(in_features=100, out_features=100, bias=True)
  (v_proj): Linear(in_features=100, out_features=100, bias=True)
  (out_proj): Linear(in_features=100, out_features=100, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

## 比较卷积神经网络、循环神经网络和自注意力

本节比较了卷积神经网络（CNN）、循环神经网络（RNN）和自注意力（Self-Attention）三种架构，目标是将一个由 $n$ 个词元组成的序列映射到另一个等长的序列，其中每个词元由 $d$ 维向量表示。比较侧重于计算复杂性、顺序操作和最大路径长度。

*   **卷积神经网络** (核大小为 $k$)：计算复杂性为 $\mathcal{O}(knd^2)$。由于其分层结构，顺序操作为 $\mathcal{O}(1)$，最大路径长度为 $\mathcal{O}(n/k)$。
*   **循环神经网络**：更新隐状态的计算复杂性为 $\mathcal{O}(d^2)$，序列总长 $n$ 导致总计算复杂性为 $\mathcal{O}(nd^2)$。具有 $\mathcal{O}(n)$ 个顺序操作和 $\mathcal{O}(n)$ 的最大路径长度，难以并行化。
*   **自注意力**：查询、键和值都是 $n \times d$ 矩阵。计算缩放点积注意力涉及 $n \times d$ 乘以 $d \times n$ (得到 $n \times n$)，再乘以 $n \times d$，因此计算复杂性为 $\mathcal{O}(n^2d)$。所有词元直接连接，顺序操作为 $\mathcal{O}(1)$，最大路径长度也为 $\mathcal{O}(1)$，有利于并行计算和捕捉远距离依赖。

总而言之，CNN 和自注意力都具有并行计算的优势，且自注意力的最大路径长度最短。然而，自注意力因为其计算复杂性与序列长度的平方成正比，在处理很长的序列时会非常慢。

## 位置编码

为了在自注意力等并行处理序列的模型中引入序列的顺序信息（因为这些模型放弃了循环神经网络的顺序操作），通过在输入表示中添加**位置编码**（positional encoding）来注入绝对或相对的位置信息。位置编码可以是学习得到的，也可以是固定的。这里描述的是一种基于正弦和余弦函数的固定位置编码。

假设输入表示 $\mathbf{X} \in \mathbb{R}^{n \times d}$ 包含 $n$ 个词元的 $d$ 维嵌入表示。位置编码使用相同形状的位置嵌入矩阵 $\mathbf{P} \in \mathbb{R}^{n \times d}$，并输出 $\mathbf{X} + \mathbf{P}$。矩阵 $\mathbf{P}$ 的第 $i$ 行、第 $2j$ 列和 $2j+1$ 列上的元素定义如下：

$$\begin{aligned} p_{i, 2j} &= \sin\left(\frac{i}{10000^{2j/d}}\right),\\p_{i, 2j+1} &= \cos\left(\frac{i}{10000^{2j/d}}\right).\end{aligned}$$

In [5]:
class PositionalEncoding(nn.Module):
    """位置编码"""
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        # 创建一个足够长的P
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(
            -1, 1) / torch.pow(10000, torch.arange(
            0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X)

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

行代表词元在序列中的位置，列代表位置编码的不同维度

绝对位置信息



沿着编码维度单调降低的频率与绝对位置信息的关系：
每个数字、每两个数字和每四个数字上的比特值 在第一个最低位、第二个最低位和第三个最低位上分别交替。

在二进制表示中，较高比特位的交替频率低于较低比特位

In [6]:
for i in range(8):
    print(f'{i}的二进制是：{i:>03b}')

0的二进制是：000
1的二进制是：001
2的二进制是：010
3的二进制是：011
4的二进制是：100
5的二进制是：101
6的二进制是：110
7的二进制是：111


相对位置信息

位置编码还允许模型学习得到输入序列中相对位置信息。 这是因为对于任何确定的位置偏移 δ ，位置 i+δ 处 的位置编码可以线性投影位置 i 处的位置编码来表示。